# Lab 15 — Controlled Concept Drift & Adaptation

*AI Cybersecurity Masterclass · Lab 15 · AI60secLab*

**Companion video:** [▶ Why Security AI Gets Dumber Over Time](https://www.youtube.com/watch?v=7BqRTjWFUBw&list=PLG4l6Ssi9A00)

**Research question:** What happens when the relationship between telemetry and the target changes after a model learned the original relationship?

Because UNSW-NB15 does not provide a real production timeline for this exercise, this notebook uses a clearly labeled controlled concept-drift simulation inside the training data.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_unsw
from src.preprocessing import split_xy
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame
from src.drift import simulate_region_concept_shift

train, _ = load_unsw(DATA_DIR)

X, y = split_xy(train, "label")
X = drop_identifier_like_columns(X)

X_pre, X_post, y_pre, y_post = train_test_split(
    X,
    y,
    test_size=0.40,
    stratify=y,
    random_state=42,
)

X_adapt, X_future, y_adapt, y_future = train_test_split(
    X_post,
    y_post,
    test_size=0.70,
    stratify=y_post,
    random_state=43,
)

In [ ]:
drift_feature = next(
    c for c in ["rate", "sbytes", "dur", "spkts"]
    if c in X.columns
)

X_adapt = X_adapt.reset_index(drop=True)
y_adapt = y_adapt.reset_index(drop=True)
X_future = X_future.reset_index(drop=True)
y_future = y_future.reset_index(drop=True)

y_adapt_shifted, adapt_meta = simulate_region_concept_shift(
    X_adapt,
    y_adapt,
    feature=drift_feature,
    quantile=0.75,
    flip_probability=0.35,
    random_state=42,
)

y_future_shifted, future_meta = simulate_region_concept_shift(
    X_future,
    y_future,
    feature=drift_feature,
    quantile=0.75,
    flip_probability=0.35,
    random_state=99,
)

print("Drift feature:", drift_feature)
print("Adaptation slice:", adapt_meta)
print("Future slice:", future_meta)

In [ ]:
baseline = logistic_pipeline(X_pre)
baseline.fit(X_pre, y_pre)

pred_before = baseline.predict(X_future)
score_before = baseline.predict_proba(X_future)[:, 1]

before_metrics = binary_metrics(
    y_future_shifted,
    pred_before,
    score_before,
)

display(
    binary_metrics_frame(
        "Pre-drift model on shifted concept",
        before_metrics,
    ).round(4)
)

In [ ]:
X_retrain = pd.concat(
    [X_pre.reset_index(drop=True), X_adapt],
    ignore_index=True,
)
y_retrain = pd.concat(
    [y_pre.reset_index(drop=True), y_adapt_shifted],
    ignore_index=True,
)

adapted = logistic_pipeline(X_retrain)
adapted.fit(X_retrain, y_retrain)

pred_after = adapted.predict(X_future)
score_after = adapted.predict_proba(X_future)[:, 1]

after_metrics = binary_metrics(
    y_future_shifted,
    pred_after,
    score_after,
)

comparison = pd.concat([
    binary_metrics_frame("Before adaptation", before_metrics),
    binary_metrics_frame("After adaptation", after_metrics),
])

display(comparison.round(4))

## Interpretation

How much performance did the simulated concept change remove? Did a relatively small labeled adaptation sample recover performance? Why is concept drift different from ordinary input-distribution shift?

**Takeaway:** production systems may need monitoring for changes in the relationship between evidence and outcomes.